In [0]:
from pyspark.sql import functions as F

bronze_category_translation = spark.table("brazilian_ecommerce.prod.bronze_category_translation")

def clean_text(col: str):
    return F.trim(F.lower(F.regexp_replace(F.col(col), "_", " ")))

staging_category_translation = (
    bronze_category_translation
    .withColumn("product_category_name", clean_text("product_category_name"))
    .withColumn("product_category_name_english", clean_text("product_category_name_english"))
    .filter(F.col("product_category_name").isNotNull())
    .dropDuplicates(["product_category_name"])
    .withColumn("_loaded_at", F.current_timestamp())
)

full_table_name = "brazilian_ecommerce.prod.staging_category_translation"

(
    staging_category_translation.write
    .mode("overwrite")
    .option("comment", "product category translation staging table")
    .option("overwriteSchema", "true")
    .saveAsTable(full_table_name)
)

# primary key
spark.sql(f"""
    ALTER TABLE {full_table_name}
    ALTER COLUMN product_category_name SET NOT NULL
""")

spark.sql(f"""
    ALTER TABLE {full_table_name}
    ADD CONSTRAINT pk_staging_category_translation PRIMARY KEY (product_category_name)
""")

print(f"Loaded {staging_category_translation.count()} rows into {full_table_name}")